# Chapter 15: Perfect Hash Filters (BPHF)

Companion notebook for **Chapter 15** of *Bernoulli Types: An Algebra of Approximate Computation*. Source files: `book/chapters/ch15/`.

Chapter 14 built the **Bernoulli Hash Function (BHF)**: an FNR-zero approximate set, perfect hash plus an $r$-bit fingerprint, with $\fprate \le 2^{-r}$ at $r = \lceil -\log_2 \fprate \rceil$ bits per element. This chapter walks the opposite axis of the perfect-filter line: the **Bernoulli Perfect Hash Filter (BPHF)**, perfect hash plus an $r$-bit *rank* table at $r = \lceil \log_2 n \rceil$ bits per element.

The chapter's honest accounting is the point of this notebook, so every claim is checked with an `assert` (execution exit code 0 means all checks passed):

1. **General query model.** A non-member presenting a recomputable hash-based rank matches the stored rank with probability about $1/n$. The bare construction *is* the BHF: $\fnrate = 0$, $\fprate \approx 2^{-r} \approx 1/n$. The rank relabel alone does not flip the error axis.
2. **Witness model.** When a non-member carries no valid rank to present, the false-positive rate is exactly 0.
3. **Exact corner.** With a true perfect hash, every member owns its slot, so $\fnrate = 0$. Rank collisions among members cause no misses.
4. **k-perfect dial.** Shrinking to $m < n$ slots crowds out members, giving $\fnrate \approx 1 - m/n$ while $\fprate$ stays 0.
5. **Ranked search.** Top-$k$ retrieval is the witness model: exact at full resolution, controlled misses under the k-perfect dial, never an off-corpus result.
6. **Space crossover.** The BPHF rank table ($n\lceil\log_2 n\rceil$ bits) and a BHF at FPR target $\varepsilon$ meet near $\varepsilon = 1/n$.

In [1]:
import hashlib
import math

import numpy as np

rng = np.random.default_rng(15)
print('NumPy', np.__version__)

def _h(salt, key, mod):
    """Salted SHA-256 of key, reduced into [0, mod). Distinct salts give independent functions."""
    d = hashlib.sha256((salt + '|' + str(key)).encode()).digest()
    return int.from_bytes(d[:8], 'big') % mod

def rho_hash(key, n):
    """Recomputable rank in [0, n): the witness a non-member could present in the general model.
    Independent of the slot hash phi (different salt)."""
    return _h('rho', key, n)

n = 1000
r = int(math.ceil(math.log2(n)))
print(f'n = {n}, r = ceil(log2 n) = {r} bits')
assert r == 10

NumPy 2.4.4
n = 1000, r = ceil(log2 n) = 10 bits


## The construction

The perfect hash `phi` is a bijection on the members onto `[n]` (a minimal perfect hash; here an index map). Its extension to non-members hashes into `[n]` with a salt independent of the rank, so a non-member lands in *some* member's slot. The rank table `R` stores `rho(a)` at `phi(a)`. The query accepts `x` when `R[phi(x)] == rho(x)`. There is one `r`-bit value per slot: the rank is itself the witness.

In [2]:
class BPHF:
    """Minimal-perfect-hash + rank table. One r-bit value per slot; the rank is the witness.
    The slot hash (phi) and the rank hash are independent (different salts)."""
    def __init__(self, members, ranks):
        self.phi = {m: i for i, m in enumerate(members)}   # minimal perfect hash: bijection onto [n]
        self.n = len(members)
        self.R = [None] * self.n
        for m in members:
            self.R[self.phi[m]] = ranks[m]

    def _phi_ext(self, x):
        if x in self.phi:
            return self.phi[x]
        return _h('phi', x, self.n)    # non-member: independent slot hash, lands in some occupied slot

    def query(self, x, rank_of):
        return self.R[self._phi_ext(x)] == rank_of(x)

members = [f'doc{i:04d}' for i in range(n)]
ranks = {m: i for i, m in enumerate(members)}   # distinct ranks 0..n-1 (a permutation)
bphf = BPHF(members, ranks)
print('built BPHF over', n, 'members,', r, 'bits per slot')

built BPHF over 1000 members, 10 bits per slot


## Claims 1 and 2: the bare construction is the BHF; the witness model gives FPR = 0

In the general model a non-member presents `rho_hash(x, n)`, independent of its slot, so the comparison succeeds with probability about `1/n`. In the witness model a non-member has no valid rank to present (modelled by a sentinel that never matches).

In [3]:
Q = 100_000

# General model: a non-member presents a hash-based rank in [0, n), independent of its slot.
fp_general = sum(bphf.query(f'x{j}', lambda z: rho_hash(z, n)) for j in range(Q))
expected = Q / n
print(f'general-model false positives: {fp_general}  (expected ~ Q/n = {expected:.0f}, i.e. FPR ~ 1/n ~ 2^-r)')
assert 0.5 * expected < fp_general < 2.0 * expected, 'general-model FPR should be ~1/n (BHF behaviour)'

# Witness model: a non-member has no valid rank (sentinel -1 never equals a stored rank in [0, n)).
fp_witness = sum(bphf.query(f'x{j}', lambda z: -1) for j in range(Q))
print(f'witness-model false positives: {fp_witness}')
assert fp_witness == 0, 'witness-model FPR must be exactly 0'

general-model false positives: 95  (expected ~ Q/n = 100, i.e. FPR ~ 1/n ~ 2^-r)
witness-model false positives: 0


## Claim 3: a true perfect hash gives FNR = 0, and rank collisions do not cause misses

In [4]:
# Every member owns its slot and reads back its own rank.
misses = sum(not bphf.query(m, lambda z: ranks[z]) for m in members)
print(f'member misses (true perfect hash): {misses}')
assert misses == 0, 'true perfect hash must miss no member'

# Force a rank collision: doc0001 takes doc0000's rank. Both must still be retrieved (distinct slots).
tie = dict(ranks); tie['doc0001'] = tie['doc0000']
bphf_tie = BPHF(members, tie)
both = bphf_tie.query('doc0000', lambda z: tie[z]) and bphf_tie.query('doc0001', lambda z: tie[z])
print(f'both rank-colliding members retrieved: {both}')
assert both, 'a rank collision must NOT cause a false negative under a perfect hash'

member misses (true perfect hash): 0
both rank-colliding members retrieved: True


## Claim 4: the k-perfect dial gives FNR = 1 - m/n at FPR = 0

A balanced k-perfect hash packs members evenly and keeps one per slot. Round-robin assignment (member `i` to slot `i % m`, first arrival kept) models the even packing: exactly `n - m` members are crowded out.

In [5]:
def kperfect_fnr(n, m):
    kept = set(); misses = 0
    for i in range(n):
        s = i % m
        if s in kept:
            misses += 1          # slot already taken: this member is crowded out
        else:
            kept.add(s)
    return misses / n

for frac in (1.00, 0.95, 0.90, 0.80):
    m = int(round(frac * n))
    fnr = kperfect_fnr(n, m)
    predicted = 1 - m / n
    print(f'm = {m:4d} ({frac:.2f} n):  FNR = {fnr:.3f}   predicted 1 - m/n = {predicted:.3f}   FPR = 0')
    assert abs(fnr - predicted) < 1e-9, 'k-perfect FNR must equal 1 - m/n'

m = 1000 (1.00 n):  FNR = 0.000   predicted 1 - m/n = 0.000   FPR = 0
m =  950 (0.95 n):  FNR = 0.050   predicted 1 - m/n = 0.050   FPR = 0
m =  900 (0.90 n):  FNR = 0.100   predicted 1 - m/n = 0.100   FPR = 0
m =  800 (0.80 n):  FNR = 0.200   predicted 1 - m/n = 0.200   FPR = 0


## Claim 5: ranked search (top-k over a corpus) is the witness model

A corpus document carries its sort-position rank; an off-corpus candidate has none to present. At full resolution the retrieval is exact; the k-perfect dial trades a controlled miss rate for a smaller table, never surfacing an off-corpus document.

In [6]:
N = 100_000
k = 100
scores = rng.uniform(size=N)
order = np.argsort(-scores, kind='stable')
top = [f'doc{int(i):06d}' for i in order[:k]]          # the true top-k documents, in rank order
top_rank = {d: i for i, d in enumerate(top)}
search = BPHF(top, top_rank)

# Full resolution: every top-k document retrieved at its true rank, no off-corpus document surfaced.
recall = sum(search.query(d, lambda z: top_rank[z]) for d in top)
off_corpus = [f'other{j}' for j in range(20_000)]      # not in the corpus, no valid rank
surfaced = sum(search.query(d, lambda z: -1) for d in off_corpus)
print(f'top-{k} recall (exact corner): {recall}/{k}   off-corpus documents surfaced: {surfaced}')
assert recall == k, 'exact corner must retrieve every top-k document'
assert surfaced == 0, 'ranked search must surface no off-corpus document (FPR = 0)'

# k-perfect dial: expected misses ~ k(1 - m/n). At m = 0.99 n, about 1 of the top 100.
for frac in (1.00, 0.99, 0.95):
    exp_miss = k * (1 - frac)
    print(f'  m = {frac:.2f} n:  expected top-{k} misses ~ k(1 - m/n) = {exp_miss:.1f}   FPR = 0')

top-100 recall (exact corner): 100/100   off-corpus documents surfaced: 0
  m = 1.00 n:  expected top-100 misses ~ k(1 - m/n) = 0.0   FPR = 0
  m = 0.99 n:  expected top-100 misses ~ k(1 - m/n) = 1.0   FPR = 0
  m = 0.95 n:  expected top-100 misses ~ k(1 - m/n) = 5.0   FPR = 0


## Claim 6: space crossover near FPR = 1/n

The BPHF spends $\lceil \log_2 n \rceil$ bits per element on the rank table (one value per slot). A BHF at FPR target $\varepsilon$ spends $\lceil -\log_2 \varepsilon \rceil$ bits. The two table widths tie at $\varepsilon = 1/n$.

In [7]:
bphf_bits = n * r
for eps in (0.1, 0.01, 1.0 / n, 1e-4):
    r_bhf = int(math.ceil(-math.log2(eps)))
    bhf_bits = n * r_bhf
    rel = 'BPHF<BHF' if bphf_bits < bhf_bits else ('equal' if bphf_bits == bhf_bits else 'BHF<BPHF')
    print(f'FPR={eps:8.5f}: BPHF {bphf_bits} bits, BHF {bhf_bits} bits  ({rel})')

r_cross = int(math.ceil(-math.log2(1.0 / n)))
print(f'crossover: r_bhf(1/n) = {r_cross}, r = {r}')
assert r_cross == r, 'BHF and BPHF widths must tie at FPR = 1/n'
print('\nAll chapter 15 claims verified.')

FPR= 0.10000: BPHF 10000 bits, BHF 4000 bits  (BHF<BPHF)
FPR= 0.01000: BPHF 10000 bits, BHF 7000 bits  (BHF<BPHF)
FPR= 0.00100: BPHF 10000 bits, BHF 10000 bits  (equal)
FPR= 0.00010: BPHF 10000 bits, BHF 14000 bits  (BPHF<BHF)
crossover: r_bhf(1/n) = 10, r = 10

All chapter 15 claims verified.
